# ਪਾਠ 10 - ਉਤਪਾਦਨ ਵਿੱਚ AI ਏਜੰਟ

ਇਸ ਪਾਠ ਵਿੱਚ ਤੁਸੀਂ Microsoft Agent Framework (Python) ਦੀ ਵਰਤੋਂ ਨਾਲ AI ਏਜੰਟਾਂ ਲਈ **ਉਤਪਾਦਨ ਪੈਟਰਨ** ਸਿੱਖੋਗੇ। ਅਸੀਂ ਕਵਰ ਕਰਦੇ ਹਾਂ:

- **ਦੇਖ-ਰੇਖ** — ਏਜੰਟ ਪਰਸਪਰਕ੍ਰਿਆਵਾਂ ਲਈ ਸਮਾਂ ਅਤੇ ਲੌਗਿੰਗ ਜੋੜਨਾ
- **ਮੁਲਾਂਕਣ** — ਪ੍ਰਤੀਕਿਰਿਆ ਗੁਣਵੱਤਾ ਨੂੰ ਸਕੋਰ ਕਰਨ ਲਈ ਇੱਕ ਮੁਲਾਂਕਣ ਏਜੰਟ ਦੀ ਵਰਤੋਂ ਕਰਨਾ
- **ਲਾਗਤ ਪ੍ਰਬੰਧਨ** — ਟੋਕਨ ਅਪਟਿਮਾਈਜੇਸ਼ਨ ਅਤੇ ਮਾਡਲ ਚੋਣ ਲਈ ਰਣਨੀਤੀਆਂ

ਦ੍ਰਿਸ਼ਟੀਕੋਣ ਇੱਕ **ਯਾਤਰਾ ਏਜੰਟ** ਹੈ ਜੋ ਉਪਭੋਗਤਾਵਾਂ ਦੀ ਯਾਤਰਾ ਯੋਜਨਾ ਬਣਾਉਣ ਵਿੱਚ ਮਦਦ ਕਰਦਾ ਹੈ, ਜਿਸ 'ਤੇ ਕੰਟਰੋਲ ਅਤੇ ਮੁਲਾਂਕਣ ਲਗਾਇਆ ਗਿਆ ਹੈ।


## ਸੈਟਅਪ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ਪ੍ਰੋਡਕਸ਼ਨ ਵਿਚਾਰ

ਏਆਈ ਏਜੰਟਾਂ ਨੂੰ ਪ੍ਰੋਟੋਟਾਈਪ ਤੋਂ ਪ੍ਰੋਡਕਸ਼ਨ ਵੱਲ ਲਿਜਾਣ ਲਈ ਤਿੰਨ ਮੁੱਖ ਸਤੰਭਾਂ 'ਤੇ ਧਿਆਨ ਦੇਣਾ ਲਾਜ਼ਮੀ ਹੈ:

1. **ਦੇਖਭਾਲ ਯੋਗਤਾ** — ਤੁਹਾਨੂੰ ਇਹ ਦੇਖਣਾ ਜਰੂਰੀ ਹੈ ਕਿ ਏਜੰਟ ਕੀ ਕਰ ਰਿਹਾ ਹੈ, ਇਸਦੇ ਕਰਮ ਕਿੰਨੇ ਸਮੇਂ ਲੈਂਦੇ ਹਨ, ਅਤੇ ਓਹ ਕਿਹੜੇ ਸੰਦ ਵਰਤਦਾ ਹੈ। ਬਿਨਾਂ ਟ੍ਰੇਸਿੰਗ ਅਤੇ ਲਾਗਿੰਗ ਦੇ, ਪ੍ਰੋਡਕਸ਼ਨ ਸਮੱਸਿਆਵਾਂ ਦੀ ਡੀਬੱਗਿੰਗ ਕਰਨਾ ਲਗਭਗ ਅਸੰਭਵ ਹੁੰਦਾ ਹੈ।

2. **ਮੂਲਾਂਕਣ** — ਆਟੋਮੇਟਿਕ ਕੁਆਲਟੀ ਚੈੱਕ ਯਕੀਨੀ ਬਣਾਉਂਦੇ ਹਨ ਕਿ ਏਜੰਟ ਦੇ ਜਵਾਬ ਸਮੇਂ ਦੇ ਨਾਲ ਸਹੀ, ਪੂਰੇ ਅਤੇ ਮਦਦਗਾਰ ਰਹਿਣ। ਇੱਕ ਮੂਲਾਂਕਣ ਏਜੰਟ ਪਰਿਭਾਸ਼ਤ ਮਾਪਦੰਡਾਂ ਦੇ ਖਿਲਾਫ ਜਵਾਬਾਂ ਨੂੰ ਸਕੋਰ ਕਰ ਸਕਦਾ ਹੈ।

3. **ਲਾਗਤ ਪ੍ਰਬੰਧਨ** — ਟੋਕਨ ਦੀ ਵਰਤੋਂ ਸਿੱਧਾ ਲਾਗਤ ਨੂੰ ਪ੍ਰਭਾਵਿਤ ਕਰਦੀ ਹੈ। ਪ੍ਰੰਪਟ ਅਪਟੀਮਾਈਜੇਸ਼ਨ, ਮਾਡਲ ਚੋਣ ਅਤੇ ਕੈਸ਼ਿੰਗ ਵਰਗੀਆਂ ਰਣਨੀਤੀਆਂ ਖ਼ਰਚੇ ਕੰਟਰੋਲ ਵਿਚ ਰੱਖਦੀਆਂ ਹਨ ਬਿਨਾਂ ਗੁਣਵੱਤਾ ਨੂੰ ਕਮਜ਼ੋਰ ਕੀਤੇ।


## ਇੱਕ ਦ੍ਰਿਸ਼ਯਮਾਨ ਏਜੰਟ ਬਣਾਉਣਾ

ਅਸੀਂ ਯਾਤਰਾ ਦੇ ਉਪਕਰਨਾਂ ਦੀ ਵਿਆਖਿਆ ਕਰਦੇ ਹਾਂ ਅਤੇ ਏਜੰਟ ਕਾਲ ਨੂੰ ਟਾਈਮਿੰਗ ਨਾਲ ਲਪੇਟਦੇ ਹਾਂ ਤਾਂ ਜੋ ਅਸੀਂ ਦੇਰੀ ਨੂੰ ਨਿਗਰਾਨੀ ਕਰ ਸਕੀਏ। ਉਤਪਾਦਨ ਵਿੱਚ ਤੁਸੀਂ OpenTelemetry ਜਾਂ ਇਸੇ ਤਰ੍ਹਾਂ ਦੇ ਟ੍ਰੇਸਿੰਗ ਬੈਕਐਂਡ ਨਾਲ ਏਕਤਾ ਕਰੋਗੇ।


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## ਮੁਲਾਂਕਣ ਦੇ ਨਮੂਨੇ

ਇੱਕ ਆਮ ਉਤਪਾਦਨ ਨਮੂਨਾ ਦੂਜੇ ਏਜੰਟ ਨੂੰ **ਮੁਲਾਂਕਣਕਾਰ** ਵਜੋਂ ਵਰਤਣਾ ਹੈ। ਮੁਲਾਂਕਣਕਾਰ ਮੁੱਖ ਏਜੰਟ ਦੇ ਜਵਾਬ ਨੂੰ ਪਹਿਲਾਂ ਤੋਂ ਨਿਰਧਾਰਤ ਮਾਪਦੰਡਾਂ ਜਿਵੇਂ ਕਿ ਪੂਰਨਤਾ, ਸਹੀਤਾ, ਅਤੇ ਮਦਦਗਾਰਤਾ ਦੇ ਖਿਲਾਫ ਅੰਕਿਤ ਕਰਦਾ ਹੈ।

ਇਸ ਨਾਲ ਸੰਭਵ ਹੁੰਦਾ ਹੈ:
- ਉਤਪਾਦਨ ਵਾਲੇ ਗੁਣਵੱਤਾ ਰੋਕਥਾਮ ਲਾਗੂ ਕੀਤੀ ਜਾਵੇ ਜਦ ਕਿਸੇ ਜਵਾਬ ਨੂੰ ਯੂਜ਼ਰਾਂ ਤਕ ਭੇਜਿਆ ਜਾਵੇ
- ਪ੍ਰਦਰਸ਼ਨ ਵਿੱਚ ਘਟਾਅ ਪਤਾ ਲਗਾਇਆ ਜਾਵੇ ਜਦ ਪ੍ਰਾਂਪਟ ਜਾਂ ਮਾਡਲ ਬਦਲਦੇ ਹਨ
- ਸਮੇਂ ਦੇ ਨਾਲ ਏਜੰਟ ਦੀ ਕਾਰਗੁਜ਼ਾਰੀ ਦੀ ਲਗਾਤਾਰ ਨਿਗਰਾਨੀ ਕੀਤੀ ਜਾਵੇ


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## ਲਾਗਤ ਪ੍ਰਬੰਧਨ ਰਣਨੀਤੀਆਂ

ਉਤਪਾਦਨ AI ਏਜੰਟਾਂ ਲਈ ਲਾਗਤਾਂ ਨੂੰ ਕੰਟਰੋਲ ਕਰਨਾ ਅਹੰਕਾਰਪੂਰਨ ਹੈ। ਇੱਥੇ ਕੁਝ ਮੁੱਖ ਰਣਨੀਤੀਆਂ ਹਨ:

| ਰਣਨੀਤੀ | ਵਰਣਨ |
|---|---|
| **ਪ੍ਰੰਪਟ ਅਪਟੀਮਾਈਜ਼ੇਸ਼ਨ** | ਸਿਸਟਮ ਨਿਰਦੇਸ਼ਾਂ ਨੂੰ ਸੰਖੇਪ ਰੱਖੋ। ਦਾਖਲ ਟੋਕਨਾਂ ਨੂੰ ਘਟਾਉਣ ਲਈ ਫਜੂਲ ਸੰਦਰਭ ਹਟਾਓ। |
    "| **ਮਾਡਲ ਚੋਣ** | ਸਾਦਾ ਕੰਮਾਂ ਲਈ ਛੋਟੇ ਤੇ ਸਸਤੇ ਮਾਡਲ ਵਰਤੋ (ਜਿਵੇਂ GPT-5-mini), ਜਿਵੇਂ ਕਿ ਕਲਾਸੀਫਿਕੇਸ਼ਨ ਜਾਂ ਨਿਕਾਸ ਲਈ, ਅਤੇ ਜਟਿਲ ਤਰੱਕੀ ਲਈ ਵੱਡੇ ਮਾਡਲ ਰੱਖੋ। |\n",
| **ਕੈਸ਼ਿੰਗ** | ਦੁਹਰਾਏ ਗਏ API ਕਾਲਾਂ ਤੋਂ ਬਚਣ ਲਈ ਟੂਲ ਨਤੀਜਿਆਂ ਤੇ ਅਕਸਰ ਪੁੱਛੇ ਜਾਣ ਵਾਲੇ ਸਵਾਲਾਂ ਨੂੰ ਕੈਸ਼ ਕਰੋ। |
| **ਟੋਕਨ ਬਜਟ** | ਅਣਜਾਣ ਲੰਮੀ ਜਵਾਬਾਂ ਨੂੰ ਰੋਕਣ ਲਈ `max_tokens` ਦੀ ਸੀਮਾ ਨਿਰਧਾਰਿਤ ਕਰੋ। |
| **ਬੈਚਿੰਗ** | ਜਿੱਥੇ ਸੰਭਵ ਹੋਵੇ ਇਕੱਲੀ API ਕਾਲ ਵਿੱਚ ਕਈ ਯੂਜ਼ਰ ਪੁੱਛਗਿੱਛਾਂ ਨੂੰ ਇਕੱਠਾ ਕਰੋ। |

ਅਮਲੀ ਤੌਰ 'ਤੇ, ਇਕ ਦਰਜੇਵਾਰ ਪদ্ধਤੀ ਚੰਗੀ ਤਰ੍ਹਾਂ ਕੰਮ ਕਰਦੀ ਹੈ: ਸਿੱਧੀਆਂ ਬੇਨਤੀਆਂ ਨੂੰ ਤੇਜ਼, ਸਸਤੇ ਮਾਡਲ ਵੱਲ ਰੁਸ਼ ਕਰਨਾ ਅਤੇ ਸਿਰਫ਼ ਜਟਿਲ ਸਵਾਲਾਂ ਨੂੰ ਵਧੀਆ ਸਮਰੱਥਾ ਵਾਲੇ ਮਾਡਲ ਨੂੰ ਭੇਜਣਾ। 


## ਸੰਖੇਪ

ਇਸ ਪਾਠ ਵਿੱਚ ਤੁਸੀਂ ਸਿੱਖਿਆ ਕਿ ਕਿਵੇਂ:

1. ਏਜੰਟ ਦੇ ਇੰਟਰੈਕਸ਼ਨਾਂ ਵਿੱਚ **ਦੇਖ-ਰੇਖ (observability)** ਸ਼ਾਮਲ ਕਰਨੀ ਹੈ ਸਮੇਂ-ਬੰਦੀ ਅਤੇ ਲੌਗਿੰਗ ਨਾਲ, ਜੋ ਟਰੇਸਿੰਗ ਅਤੇ ਨਿਗਰਾਨੀ ਲਈ ਆਧਾਰ ਤਿਆਰ ਕਰਦੀ ਹੈ।
2. **ਏਜੰਟ ਦੇ ਜਵਾਬਾਂ ਦਾ ਮੁਆਇਨਾ** ਕਰਨਾ ਆਪਣੇ ਆਪ ਇੱਕ ਮੂਲਾਂਕਣ ਏਜੰਟ ਦੀ ਵਰਤੋਂ ਨਾਲ ਜੋ ਪੂਰਨਤਾ, ਸਹੀ ਹੋਣਾ ਅਤੇ ਮਦਦਗਾਰਤਾ ਦਾ ਅੰਕ ਦਰਜ ਕਰਦਾ ਹੈ।
3. **ਲਾਗਤਾਂ ਦਾ ਪ੍ਰਬੰਧਨ** ਕਰਨਾ prompt ਨੂੰ ਸੁਧਾਰ ਕੇ, ਮਾਡਲ ਚੋਣ, caching ਅਤੇ ਟੋਕਨ ਬਜਟਾਂ ਦੇ ਜਰੀਏ।

ਇਹ ਉਤਪਾਦਨ ਪੈਟਰਨ ਤੁਹਾਡੇ AI ਏਜੰਟਾਂ ਨੂੰ ਭਰੋਸੇਯੋਗ, ਮਾਪਣਯੋਗ ਅਤੇ ਖ਼ਰਚੀਲਾ-ਸਮਰੱਥ ਬਣਾਉਣ ਵਿੱਚ ਮਦਦਗਾਰ ਹਨ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ਅਸਵੀਕਾਰੋਪਣ**:
ਇਸ ਦਸਤਾਵੇਜ਼ ਦਾ ਅਨੁਵਾਦ ਏਆਈ ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਦੋਂ ਕਿ ਅਸੀਂ ਸਹੀਤਾਵਾਂ ਲਈ ਯਤਨਸ਼ੀਲ ਹਾਂ, ਕਿਰਪਾ ਕਰਕੇ ਧਿਆਨ ਰੱਖੋ ਕਿ ਸਵੈਚਾਲਿਤ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਸਮੱਤਿਆਵਾਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਦਸਤਾਵੇਜ਼ ਆਪਣੀ ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਅਧਿਕਾਰਕ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਜਰੂਰੀ ਜਾਣਕਾਰੀ ਲਈ, ਪੇਸ਼ੇਵਰ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫ਼ਾਰਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਅਸੀਂ ਇਸ ਅਨੁਵਾਦ ਦੇ ਉਪਯੋਗ ਤੋਂ ਪੈਦਾ ਹੋਣ ਵਾਲੀਆਂ ਕਿਸੇ ਵੀ ਗਲਤਫਹਿਮੀਆਂ ਜਾਂ ਗਲਤ ਵਿਆਖਿਆਵਾਂ ਲਈ ਜਵਾਬਦੇਹ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
